
# Task 3 - Modular Logged ETL Pipeline

This notebook implements a production-style ETL pipeline using the TVMaze API.

### Features Included
- Modular ETL functions:
  - `extract()`
  - `clean()`
  - `transform()`
  - `load()`
- Full logging support
- Error handling
- Data cleaning
- Feature engineering
- Summary table using `groupby()`
- CSV export
- MySQL loading with idempotency


In [6]:

import requests
import pandas as pd
import logging
import mysql.connector
from mysql.connector import Error
import os

# ---------------- LOGGING CONFIG ----------------
logging.basicConfig(
    filename='etl_pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logger = logging.getLogger()

API_URL = "https://api.tvmaze.com/shows"
CSV_FILE = "tvmaze_cleaned.csv"

# ---------------- MYSQL CONFIG ----------------
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = os.getenv("DB_PASSWORD")
MYSQL_DATABASE = "etl_project"
MYSQL_TABLE = "tvmaze_shows"


## 1. Extract Function

In [7]:

def extract(url, timeout=10):
    logger.info("Starting extraction process")

    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()

        data = response.json()

        df = pd.DataFrame(data)

        logger.info(f"Extraction completed successfully. Rows extracted: {len(df)}")

        return df

    except requests.exceptions.Timeout:
        logger.error("Request timed out")

    except requests.exceptions.HTTPError as e:
        logger.error(f"HTTP Error: {e}")

    except requests.exceptions.ConnectionError:
        logger.error("Connection Error")

    except Exception as e:
        logger.error(f"Unexpected Error: {e}")

    return pd.DataFrame()

# Example usage of the extract function
API_URL = 'https://api.tvmaze.com/shows'
raw_df = extract(API_URL)

if not raw_df.empty:
    print("Extracted Data Head from TVMaze API:")
    display(raw_df.head())
else:
    print("No data extracted from TVMaze API.")

Extracted Data Head from TVMaze API:


,id,url,name,type,language,genres,status,runtime,averageRuntime,premiered,...,rating,weight,network,webChannel,dvdCountry,externals,image,summary,updated,_links
0,1,https://www.tvmaze.com/shows/1/under-the-dome,Under the Dome,Scripted,English,"[Drama, Science-Fiction, Thriller]",Ended,60.0,60,2013-06-24,...,{'average': 6.6},100,"{'id': 2, 'name': 'CBS', 'country': {'name': '...",None,None,"{'tvrage': 25988, 'thetvdb': 264492, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p><b>Under the Dome</b> is the story of a sma...,1769177765,{'self': {'href': 'https://api.tvmaze.com/show...
1,2,https://www.tvmaze.com/shows/2/person-of-interest,Person of Interest,Scripted,English,"[Action, Crime, Science-Fiction]",Ended,60.0,60,2011-09-22,...,{'average': 8.8},100,"{'id': 2, 'name': 'CBS', 'country': {'name': '...",None,None,"{'tvrage': 28376, 'thetvdb': 248742, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p>You are being watched. The government has a...,1778904153,{'self': {'href': 'https://api.tvmaze.com/show...
2,3,https://www.tvmaze.com/shows/3/bitten,Bitten,Scripted,English,"[Drama, Horror, Romance]",Ended,60.0,60,2014-01-11,...,{'average': 7.4},96,"{'id': 7, 'name': 'CTV Sci-Fi Channel', 'count...",None,None,"{'tvrage': 34965, 'thetvdb': 269550, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p>Based on the critically acclaimed series of...,1704793709,{'self': {'href': 'https://api.tvmaze.com/show...
3,4,https://www.tvmaze.com/shows/4/arrow,Arrow,Scripted,English,"[Drama, Action, Science-Fiction]",Ended,60.0,60,2012-10-10,...,{'average': 7.4},99,"{'id': 5, 'name': 'The CW', 'country': {'name'...",None,None,"{'tvrage': 30715, 'thetvdb': 257655, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,"<p>After a violent shipwreck, billionaire play...",1766868900,{'self': {'href': 'https://api.tvmaze.com/show...
4,5,https://www.tvmaze.com/shows/5/true-detective,True Detective,Scripted,English,"[Drama, Crime, Thriller]",Running,60.0,63,2014-01-12,...,{'average': 8.1},100,"{'id': 8, 'name': 'HBO', 'country': {'name': '...",None,None,"{'tvrage': 31369, 'thetvdb': 270633, 'imdb': '...",{'medium': 'https://static.tvmaze.com/uploads/...,<p>Touch darkness and darkness touches you bac...,1777231753,{'self': {'href': 'https://api.tvmaze.com/show...


## 2. Clean Function

In [8]:

def clean(df):
    logger.info(f"Cleaning started. Input rows: {len(df)}")

    if df.empty:
        logger.warning("Empty DataFrame received in clean()")
        return pd.DataFrame()

    cleaned_df = df.copy()

    # ---------------- NORMALIZE NESTED COLUMNS ----------------
    cleaned_df['rating_average'] = cleaned_df['rating'].apply(
        lambda x: x.get('average') if isinstance(x, dict) else None
    )

    cleaned_df['schedule_time'] = cleaned_df['schedule'].apply(
        lambda x: x.get('time') if isinstance(x, dict) else None
    )

    cleaned_df['schedule_days'] = cleaned_df['schedule'].apply(
        lambda x: ', '.join(x.get('days')) if isinstance(x, dict) else None
    )

    # ---------------- REMOVE UNUSED COLUMNS ----------------
    drop_columns = [
        'runtime',
        'averageRuntime',
        'officialSite',
        'weight',
        'webChannel',
        'updated',
        'rating',
        'schedule',
        'network',
        'image',
        '_links',
        'externals',
        'dvdCountry'
    ]

    existing_cols = [col for col in drop_columns if col in cleaned_df.columns]

    cleaned_df.drop(columns=existing_cols, inplace=True)

    # ---------------- HANDLE NULL VALUES ----------------
    cleaned_df['rating_average'] = cleaned_df['rating_average'].fillna(0)

    object_columns = cleaned_df.select_dtypes(include='object').columns

    for col in object_columns:
        cleaned_df[col] = cleaned_df[col].fillna('Unknown')

    # ---------------- DATE CONVERSION ----------------
    for col in ['premiered', 'ended']:
        if col in cleaned_df.columns:
            cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

    # ---------------- REMOVE HTML TAGS ----------------
    if 'summary' in cleaned_df.columns:
        cleaned_df['summary'] = cleaned_df['summary'].str.replace(
            r'<.*?>', '',
            regex=True
        )

    logger.info(f"Cleaning completed. Output rows: {len(cleaned_df)}")

    return cleaned_df

# Example usage of the clean function
cleaned_df = clean(raw_df)

if not cleaned_df.empty:
    print("\nCleaned Data Head:")
    display(cleaned_df.head())
    print("\nCleaned Data Info:")
    cleaned_df.info()
else:
    print("No data after cleaning.")



Cleaned Data Head:


C:\Users\sumit\AppData\Local\Temp\ipykernel_22520\2454680394.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = cleaned_df.select_dtypes(include='object').columns


,id,url,name,type,language,genres,status,premiered,ended,summary,rating_average,schedule_time,schedule_days
0,1,https://www.tvmaze.com/shows/1/under-the-dome,Under the Dome,Scripted,English,"[Drama, Science-Fiction, Thriller]",Ended,2013-06-24,2015-09-10,Under the Dome is the story of a small town th...,6.6,22:00,Thursday
1,2,https://www.tvmaze.com/shows/2/person-of-interest,Person of Interest,Scripted,English,"[Action, Crime, Science-Fiction]",Ended,2011-09-22,2016-06-21,You are being watched. The government has a se...,8.8,22:00,Tuesday
2,3,https://www.tvmaze.com/shows/3/bitten,Bitten,Scripted,English,"[Drama, Horror, Romance]",Ended,2014-01-11,2016-04-15,Based on the critically acclaimed series of no...,7.4,22:00,Friday
3,4,https://www.tvmaze.com/shows/4/arrow,Arrow,Scripted,English,"[Drama, Action, Science-Fiction]",Ended,2012-10-10,2020-01-28,"After a violent shipwreck, billionaire playboy...",7.4,21:00,Tuesday
4,5,https://www.tvmaze.com/shows/5/true-detective,True Detective,Scripted,English,"[Drama, Crime, Thriller]",Running,2014-01-12,NaT,Touch darkness and darkness touches you back. ...,8.1,21:00,Sunday



Cleaned Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id              240 non-null    int64         
 1   url             240 non-null    str           
 2   name            240 non-null    str           
 3   type            240 non-null    str           
 4   language        240 non-null    str           
 5   genres          240 non-null    object        
 6   status          240 non-null    str           
 7   premiered       240 non-null    datetime64[us]
 8   ended           219 non-null    datetime64[us]
 9   summary         240 non-null    str           
 10  rating_average  240 non-null    float64       
 11  schedule_time   240 non-null    str           
 12  schedule_days   240 non-null    str           
dtypes: datetime64[us](2), float64(1), int64(1), object(1), str(8)
memory usage: 24.5+ KB


## 3. Transform Function

In [9]:

def transform(df):
    logger.info(f"Transformation started. Input rows: {len(df)}")

    if df.empty:
        logger.warning("Empty DataFrame received in transform()")
        return pd.DataFrame(), pd.DataFrame()

    transformed_df = df.copy()

    # ---------------- FEATURE ENGINEERING ----------------

    # 1. Premier Year
    transformed_df['premier_year'] = transformed_df['premiered'].dt.year.fillna(0)

    # 2. Rating Category
    transformed_df['rating_category'] = transformed_df['rating_average'].apply(
        lambda x: (
            'Excellent' if x >= 8 else
            'Good' if x >= 6 else
            'Average' if x >= 4 else
            'Poor'
        )
    )

    # 3. Summary Word Count
    transformed_df['summary_word_count'] = transformed_df['summary'].apply(
        lambda x: len(str(x).split())
    )

    # 4. Show Duration
    transformed_df['show_duration_years'] = (
        transformed_df['ended'].dt.year.fillna(
            pd.Timestamp.now().year
        ) - transformed_df['premiered'].dt.year.fillna(0)
    )

    logger.info("Feature engineering completed")

    # ---------------- SUMMARY TABLE ----------------
    summary_table = transformed_df.groupby('language').agg(
        average_rating=('rating_average', 'mean'),
        min_rating=('rating_average', 'min'),
        max_rating=('rating_average', 'max'),
        total_shows=('id', 'count')
    ).reset_index()

    logger.info("Summary table generated")

    logger.info(f"Transformation completed. Output rows: {len(transformed_df)}")

    return transformed_df, summary_table

# Example usage of the transform function
transformed_df, show_summary = transform(cleaned_df)

if not transformed_df.empty:
    print("\nTransformed Data Head (with new columns):")
    display(transformed_df.head())
    print("\nTransformed Data Info:")
    transformed_df.info()

if show_summary is not None and not show_summary.empty:
    print("\nSummary Table by Language:")
    display(show_summary.head())
else:
    print("No data after transformation or summary table could not be generated.")




Transformed Data Head (with new columns):


,id,url,name,type,language,genres,status,premiered,ended,summary,rating_average,schedule_time,schedule_days,premier_year,rating_category,summary_word_count,show_duration_years
0,1,https://www.tvmaze.com/shows/1/under-the-dome,Under the Dome,Scripted,English,"[Drama, Science-Fiction, Thriller]",Ended,2013-06-24,2015-09-10,Under the Dome is the story of a small town th...,6.6,22:00,Thursday,2013,Good,57,2.0
1,2,https://www.tvmaze.com/shows/2/person-of-interest,Person of Interest,Scripted,English,"[Action, Crime, Science-Fiction]",Ended,2011-09-22,2016-06-21,You are being watched. The government has a se...,8.8,22:00,Tuesday,2011,Excellent,96,5.0
2,3,https://www.tvmaze.com/shows/3/bitten,Bitten,Scripted,English,"[Drama, Horror, Romance]",Ended,2014-01-11,2016-04-15,Based on the critically acclaimed series of no...,7.4,22:00,Friday,2014,Good,75,2.0
3,4,https://www.tvmaze.com/shows/4/arrow,Arrow,Scripted,English,"[Drama, Action, Science-Fiction]",Ended,2012-10-10,2020-01-28,"After a violent shipwreck, billionaire playboy...",7.4,21:00,Tuesday,2012,Good,85,8.0
4,5,https://www.tvmaze.com/shows/5/true-detective,True Detective,Scripted,English,"[Drama, Crime, Thriller]",Running,2014-01-12,NaT,Touch darkness and darkness touches you back. ...,8.1,21:00,Sunday,2014,Excellent,47,12.0



Transformed Data Info:
<class 'pandas.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id                   240 non-null    int64         
 1   url                  240 non-null    str           
 2   name                 240 non-null    str           
 3   type                 240 non-null    str           
 4   language             240 non-null    str           
 5   genres               240 non-null    object        
 6   status               240 non-null    str           
 7   premiered            240 non-null    datetime64[us]
 8   ended                219 non-null    datetime64[us]
 9   summary              240 non-null    str           
 10  rating_average       240 non-null    float64       
 11  schedule_time        240 non-null    str           
 12  schedule_days        240 non-null    str           
 13  premier_year         2

,language,average_rating,min_rating,max_rating,total_shows
0,English,7.452966,0.0,9.2,236
1,Japanese,7.850000,7.3,8.8,4


## 4. Load Function

In [10]:

def load(df, csv_file, table_name):
    logger.info(f"Loading started. Input rows: {len(df)}")

    if df.empty:
        logger.warning("Empty DataFrame received in load()")
        return

    # ---------------- SAVE CSV ----------------
    try:
        df.to_csv(csv_file, index=False)

        logger.info(f"CSV saved successfully: {csv_file}")

    except Exception as e:
        logger.error(f"CSV Save Error: {e}")

    # ---------------- MYSQL LOAD ----------------
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD
        )

        cursor = conn.cursor()

        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}")
        cursor.execute(f"USE {MYSQL_DATABASE}")

        # Create table
        create_table_query = f'''
        CREATE TABLE IF NOT EXISTS {table_name} (
            id INT PRIMARY KEY,
            name VARCHAR(255),
            language VARCHAR(100),
            status VARCHAR(100),
            genres TEXT,
            premiered DATETIME,
            ended DATETIME,
            rating_average FLOAT,
            rating_category VARCHAR(50),
            summary_word_count INT,
            show_duration_years INT
        )
        '''

        cursor.execute(create_table_query)

        rows_inserted = 0

        insert_query = f'''
        INSERT IGNORE INTO {table_name}
        (
            id,
            name,
            language,
            status,
            genres,
            premiered,
            ended,
            rating_average,
            rating_category,
            summary_word_count,
            show_duration_years
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        '''

        for _, row in df.iterrows():
            values = (
                int(row['id']),
                str(row.get('name', '')),
                str(row.get('language', '')),
                str(row.get('status', '')),
                str(row.get('genres', '')),
                row.get('premiered'),
                row.get('ended'),
                float(row.get('rating_average', 0)),
                str(row.get('rating_category', '')),
                int(row.get('summary_word_count', 0)),
                int(row.get('show_duration_years', 0))
            )

            cursor.execute(insert_query, values)

            rows_inserted += cursor.rowcount

        conn.commit()

        logger.info(f"MySQL loading completed. Rows inserted: {rows_inserted}")

    except Error as e:
        logger.error(f"MySQL Error: {e}")

    finally:
        try:
            cursor.close()
            conn.close()
            logger.info("MySQL connection closed")
        except:
            pass


## 5. Run ETL Pipeline

In [11]:

def run_pipeline():
    logger.info("ETL Pipeline Started")

    # Extract
    raw_df = extract(API_URL)

    if raw_df.empty:
        logger.error("Pipeline stopped during extraction")
        return

    # Clean
    cleaned_df = clean(raw_df)

    if cleaned_df.empty:
        logger.error("Pipeline stopped during cleaning")
        return

    # Transform
    transformed_df, summary_df = transform(cleaned_df)

    if transformed_df.empty:
        logger.error("Pipeline stopped during transformation")
        return

    print("Summary Table")
    print(summary_df.head())

    # Load
    load(transformed_df, CSV_FILE, MYSQL_TABLE)

    logger.info("ETL Pipeline Completed")

    print("ETL Pipeline Completed Successfully")
    print("Log file created: etl_pipeline.log")


In [12]:

# Execute pipeline
run_pipeline()


C:\Users\sumit\AppData\Local\Temp\ipykernel_22520\2454680394.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = cleaned_df.select_dtypes(include='object').columns


Summary Table
   language  average_rating  min_rating  max_rating  total_shows
0   English        7.452966         0.0         9.2          236
1  Japanese        7.850000         7.3         8.8            4
ETL Pipeline Completed Successfully
Log file created: etl_pipeline.log
